### SparseSimulator

This demo compares performance of cirq's built-in dense state simulator (cirq.Simulator) and my SparseSimulator.

In [1]:
from collections.abc import Iterator, Sequence

import cirq
import random
import time

from sparse_sim import SparseSimulator


def maj(a: cirq.Qid, b: cirq.Qid, c: cirq.Qid) -> Iterator[cirq.Operation]:
    yield cirq.CNOT(c, a)
    yield cirq.CNOT(c, b)
    yield cirq.CCNOT(a, b, c)


def uma(a: cirq.Qid, b: cirq.Qid, c: cirq.Qid) -> Iterator[cirq.Operation]:
    yield cirq.CCNOT(a, b, c)
    yield cirq.CNOT(c, a)
    yield cirq.CNOT(a, b)


class CuccaroAdder(cirq.Gate):
    """Adds two n-bit unsigned integers using the Cuccaro ripple-carry circuit.

    With little-endian registers, computes
    (a, b, z) -> (a, (a + b) mod 2**n, z XOR floor((a + b) / 2**n)).

    Reference: Cuccaro, Draper, Kutin, and Moulton, "A new quantum
    ripple-carry addition circuit" (2004), https://arxiv.org/abs/quant-ph/0410184
    """

    def __init__(self, n: int) -> None:
        if n < 1:
            raise ValueError("n must be positive")
        self.n = n

    @property
    def input_sizes(self) -> tuple[int, int, int]:
        # First addend, second addend, carry output.
        return self.n, self.n, 1

    def _num_qubits_(self) -> int:
        return 2 * self.n + 1

    def _decompose_with_context_(
        self,
        qubits: Sequence[cirq.Qid],
        context: cirq.DecompositionContext,
    ) -> Iterator[cirq.OP_TREE]:
        a = qubits[: self.n]
        b = qubits[self.n : 2 * self.n]
        z = qubits[2 * self.n]
        ancilla = context.qubit_manager.qalloc(1)
        c = ancilla[0]

        try:
            yield from maj(c, b[0], a[0])
            for i in range(1, self.n):
                yield from maj(a[i - 1], b[i], a[i])
            yield cirq.CNOT(a[-1], z)
            for i in range(self.n - 1, 0, -1):
                yield from uma(a[i - 1], b[i], a[i])
            yield from uma(c, b[0], a[0])
        finally:
            context.qubit_manager.qfree(ancilla)




rng = random.Random(0)
simulator = SparseSimulator()
#simulator = cirq.Simulator()

def run_cuccaro_adder(n, sim):  
    a = cirq.LineQubit.range(n)
    b = cirq.LineQubit.range(n, 2 * n)
    z = cirq.LineQubit(2 * n)
    qubits = (*a, *b, z)
    adder = CuccaroAdder(n)

    a_value = rng.randint(0, 2**n-1)
    b_value = rng.randint(0, 2**n-1)
    expected_sum = a_value + b_value

    circuit = cirq.Circuit(
        cirq.X(qubit)
        for register_value, register in ((a_value, a), (b_value, b))
        for bit, qubit in enumerate(register)
        if (register_value >> bit) & 1
    )
    circuit.append(adder.on(*qubits))
    circuit.append(
        [
            cirq.measure(*a, key="a"),
            cirq.measure(*b, key="sum"),
            cirq.measure(z, key="carry_out"),
        ]
    )

    t0 = time.perf_counter()
    result = sim.run(circuit, repetitions=1)
    time_elapsed = time.perf_counter()-t0
    sim_name = sim.__class__.__name__
    print(f"{sim_name}: {n=}, {time_elapsed=:.6f}")
    
    measured_a = sum(int(bit) << i for i, bit in enumerate(result.measurements["a"][0]))
    measured_sum = sum(int(bit) << i for i, bit in enumerate(result.measurements["sum"][0]))
    measured_carry = int(result.measurements["carry_out"][0, 0])
    assert measured_a == a_value
    assert measured_sum == expected_sum % 2**n
    assert measured_carry == expected_sum // 2**n

for n in [3,4,5]:
    run_cuccaro_adder(n, cirq.Simulator())

for n in [10, 100, 10**3, 10**4, 5 * 10**4]:
    run_cuccaro_adder(n, SparseSimulator())

Simulator: n=3, time_elapsed=0.005978
Simulator: n=4, time_elapsed=0.042375
Simulator: n=5, time_elapsed=1.041485
SparseSimulator: n=10, time_elapsed=0.000767
SparseSimulator: n=100, time_elapsed=0.003140
SparseSimulator: n=1000, time_elapsed=0.032048
SparseSimulator: n=10000, time_elapsed=0.427906
SparseSimulator: n=50000, time_elapsed=4.819097


In [2]:
def build_qft_circuit(n, input_value):
    qubits = cirq.LineQubit.range(n)
    circuit = cirq.Circuit(
        cirq.X(qubit)
        for bit, qubit in enumerate(qubits)
        if (input_value >> bit) & 1
    )
    circuit.append(cirq.qft(*qubits))
    return cirq.Circuit(cirq.decompose(circuit)), qubits


def benchmark_qft(n, sim):
    input_value = rng.randint(0, 2**n-1)
    circuit, qubits = build_qft_circuit(n, input_value)
    start = time.perf_counter()
    result = simulator.simulate(circuit, qubit_order=qubits)
    time_elapsed = time.perf_counter() - start

    sim_name = sim.__class__.__name__
    print(f"{sim_name}: {n=}, {time_elapsed=:.6f}")
    return result.final_state_vector

for n in [4, 6, 8, 10, 12, 14, 16, 17]:
    benchmark_qft(n, cirq.Simulator())
    benchmark_qft(n, SparseSimulator())


Simulator: n=4, time_elapsed=0.001794
SparseSimulator: n=4, time_elapsed=0.002759
Simulator: n=6, time_elapsed=0.002388
SparseSimulator: n=6, time_elapsed=0.002481
Simulator: n=8, time_elapsed=0.014807
SparseSimulator: n=8, time_elapsed=0.006620
Simulator: n=10, time_elapsed=0.030729
SparseSimulator: n=10, time_elapsed=0.031616
Simulator: n=12, time_elapsed=0.139986
SparseSimulator: n=12, time_elapsed=0.112313
Simulator: n=14, time_elapsed=0.497365
SparseSimulator: n=14, time_elapsed=0.507166
Simulator: n=16, time_elapsed=2.030843
SparseSimulator: n=16, time_elapsed=2.101261
Simulator: n=17, time_elapsed=6.624044
SparseSimulator: n=17, time_elapsed=6.571864
